# Credit Risk Demo - Train Scikit-learn Model

This notebook trains a Random Forest classifier using **all available features**.

**Algorithm Strengths:** Random Forest provides robust predictions through ensemble of decision trees with bootstrap aggregating (bagging).

## Steps
1. Load preprocessed data (all features)
2. Train Random Forest model
3. Evaluate performance
4. Save model as `sklearn-model.joblib`
5. Upload to MinIO S3

In [ ]:
# Install required packages
!pip install -q scikit-learn pandas numpy matplotlib seaborn minio joblib

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    roc_auc_score,
    log_loss,
    roc_curve,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from minio import Minio
from urllib.parse import urlparse
import os

print("✓ Libraries imported successfully")

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed data from data preparation notebook
X_train = np.load('data/processed/X_train.npy')
X_test = np.load('data/processed/X_test.npy')
y_train = np.load('data/processed/y_train.npy')
y_test = np.load('data/processed/y_test.npy')

# Load feature names
with open('data/processed/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"\nAll features:")
for f in feature_names:
    print(f"  - {f}")
print(f"\nClass distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")

In [ ]:
# Train Random Forest
print("Training Random Forest model...")

model = RandomForestClassifier(
    # Forest size
    n_estimators=300,              # Number of trees in the forest (more trees for better ensemble)
    
    # Tree structure - controls individual tree complexity
    max_depth=12,                  # Maximum depth of each tree (increased for more capacity)
    min_samples_split=10,          # Minimum samples required to split a node
    min_samples_leaf=5,            # Minimum samples required at leaf node
    
    # Randomness for diversity
    max_features='sqrt',           # Number of features to consider for best split
    bootstrap=True,                # Use bootstrap samples
    
    # Class imbalance handling
    class_weight='balanced',       # Automatically adjusts weights inversely proportional to class frequencies
    
    # Other
    random_state=42,
    n_jobs=-1                      # Use all CPU cores
)

model.fit(X_train, y_train)

print("✓ Training complete!")
print(f"Number of trees: {model.n_estimators}")
print(f"Max depth: {model.max_depth}")
print(f"Class weight 'balanced': Automatically computed based on class frequencies")

## 3. Evaluate Model Performance

**Note on Classification Threshold:** We use 0.5 as the decision threshold (probability > 0.5 = default). In production, this threshold should be optimized based on business costs (e.g., cost of missed default vs. cost of rejected good customer).

In [ ]:
# Make predictions on both train and test sets
y_pred_proba_train = model.predict_proba(X_train)[:, 1]
y_pred_train = model.predict(X_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

# Calculate metrics for both sets
train_accuracy = accuracy_score(y_train, y_pred_train)
train_auc = roc_auc_score(y_train, y_pred_proba_train)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("="*60)
print("RANDOM FOREST MODEL PERFORMANCE")
print("="*60)
print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  ROC AUC:   {auc:.4f}")
print(f"\nTrain vs Test Comparison (checking for overfitting):")
print(f"  Train Accuracy: {train_accuracy:.4f}")
print(f"  Test Accuracy:  {accuracy:.4f}")
print(f"  Train AUC:      {train_auc:.4f}")
print(f"  Test AUC:       {auc:.4f}")
if train_auc - auc > 0.05:
    print(f"  ⚠ Significant gap suggests overfitting")
else:
    print(f"  ✓ Good generalization")
print("="*60)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False)
plt.title('Random Forest - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='purple', lw=2, label=f'ROC curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest - ROC Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(importances.head(10))

# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = importances.head(15)
plt.barh(range(len(top_features)), top_features['importance'], color='purple')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance (Gini Importance)')
plt.title('Random Forest - Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Save Model

Save the trained Random Forest model in joblib format for MLServer deployment.

In [ ]:
# Create sklearn model directory matching MinIO structure
os.makedirs('models/sklearn', exist_ok=True)

# Save model as .joblib format (MLServer compatible)
model_path = 'models/sklearn/model.joblib'
joblib.dump(model, model_path)

print(f"✓ Model saved to: {model_path}")
print(f"  File size: {os.path.getsize(model_path) / 1024:.2f} KB")
print(f"\nRandom Forest Model:")
print(f"  - {model.n_estimators} trees")
print(f"  - Max depth: {model.max_depth}")
print(f"  - Ready for MLServer deployment")

## 5. Upload to MinIO

Upload the trained model to MinIO S3 storage for MLServer to access.

In [ ]:
# MinIO configuration from environment variables
MINIO_ENDPOINT = os.getenv('AWS_S3_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('AWS_ACCESS_KEY_ID')
MINIO_SECRET_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')
MINIO_BUCKET = os.getenv('AWS_S3_BUCKET')

# Parse endpoint to extract protocol and hostname
MINIO_SECURE = False
if MINIO_ENDPOINT:
    parsed = urlparse(MINIO_ENDPOINT if '//' in MINIO_ENDPOINT else f'//{MINIO_ENDPOINT}')
    MINIO_SECURE = parsed.scheme == 'https'
    MINIO_ENDPOINT = parsed.netloc

# Validate required environment variables
required_vars = {
    'AWS_S3_ENDPOINT': MINIO_ENDPOINT,
    'AWS_ACCESS_KEY_ID': MINIO_ACCESS_KEY,
    'AWS_SECRET_ACCESS_KEY': MINIO_SECRET_KEY,
    'AWS_S3_BUCKET': MINIO_BUCKET
}

missing_vars = [var for var, value in required_vars.items() if not value]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

print("MinIO Configuration:")
print(f"  Endpoint: {MINIO_ENDPOINT}")
print(f"  Bucket: {MINIO_BUCKET}")
print(f"  Secure: {MINIO_SECURE}")

In [ ]:
# Initialize MinIO client
try:
    client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=MINIO_SECURE
    )
    print("✓ MinIO client initialized")
except Exception as e:
    print(f"✗ Error initializing MinIO client: {e}")
    raise

In [ ]:
# Check if bucket exists
try:
    if not client.bucket_exists(MINIO_BUCKET):
        client.make_bucket(MINIO_BUCKET)
        print(f"✓ Created bucket: {MINIO_BUCKET}")
    else:
        print(f"✓ Bucket exists: {MINIO_BUCKET}")
except Exception as e:
    print(f"✗ Error checking/creating bucket: {e}")
    raise

In [ ]:
# Upload model to MinIO
try:
    print(f"Uploading sklearn model to {MINIO_BUCKET}/sklearn/...")
    
    client.fput_object(MINIO_BUCKET, 'sklearn/model.joblib', 'models/sklearn/model.joblib')
    
    print(f"✓ Model uploaded successfully!")
    print(f"  Location: s3://{MINIO_BUCKET}/sklearn/model.joblib")
    
except Exception as e:
    print(f"✗ Error uploading: {e}")
    raise

In [ ]:
# Verify upload
print("\nVerifying upload...")
try:
    stat = client.stat_object(MINIO_BUCKET, 'sklearn/model.joblib')
    print(f"✓ Model verified in MinIO:")
    print(f"  Size: {stat.size / 1024:.2f} KB")
    print(f"  Modified: {stat.last_modified}")
except Exception as e:
    print(f"✗ Error verifying: {e}")

## Summary

Scikit-learn Random Forest model training complete!

### Model Performance
- Check the metrics printed above

### Model Storage
- **Local**: `models/sklearn/model.joblib`
- **MinIO**: `s3://models/sklearn/model.joblib`

### Deployment Configuration
To get probabilities instead of class labels, configure the InferenceService with:
```yaml
env:
  - name: MLSERVER_MODEL_EXTRA
    value: '{"predict_fn": "predict_proba"}'
```

**Note:** Sklearn's `predict_proba` returns `[P(no default), P(default)]` - the ensemble app will extract the second value.

### Model Characteristics
- **Algorithm**: Random Forest (bagging ensemble of decision trees)
- **Ensemble diversity**: Different from XGBoost/LightGBM (bagging vs boosting)
- **No confounding issues**: Tree-based model handles non-linear relationships correctly

### All Three Models Trained!
You now have all three models uploaded to MinIO:
1. ✓ `xgboost/model.ubj` - Gradient boosting (sequential trees)
2. ✓ `lightgbm/model.bst` - Gradient boosting (sequential trees, categorical handling)
3. ✓ `sklearn/model.joblib` - Random Forest (parallel trees, bagging)

### Next Steps
1. Create MLServer ServingRuntime resources for each model type
2. Create InferenceService resources pointing to MinIO model paths
3. Deploy models with `MLSERVER_MODEL_EXTRA='{"predict_fn": "predict_proba"}'` for sklearn
4. Test deployed models
5. Build ensemble scoring logic